# Assemble matrices from teaseq dataset to create mudata for analysis in python


# Load Libraries


In [ ]:
quiet_library <- function(...) {
  suppressPackageStartupMessages(library(...))
}
quiet_library("H5weaver")
quiet_library("purrr")
quiet_library("dplyr")
quiet_library("Seurat")
quiet_library("ArchR")
quiet_library("MuDataSeurat")


In [ ]:
# define file path
data_path <- "/home/workspace/data/preRA_teaseq/EXP-00243"
output_path <- "/home/workspace/data/preRA_teaseq/EXP-00243/totalVI"
# define a project name
proj_name <- "preRA_teaseq_totalvi"


# Load Seurat object to extract the adt and RNA information


In [2]:
hise::cacheFiles(list("5575714a-0e62-4bfc-8b2f-e8bd8c31c5b1"))


In [ ]:
# load the teaseq seurat object which contains rna and adt information
ra_tea_so <- readRDS(file.path(data_path, "PreRA_teaseq_seurat_qc_filtered_cells_lsi.rds"))


In [ ]:
# define the matrix name to get data from
adt_mx_name <- "cleanadt"
rna_mx_name <- "RNA"


In [ ]:
# # pull adt matrix, subset by markers and re-inject under new assay identity
# adt_mtx <- ra_tea_so@assays$cleanadt@counts %>% as.data.frame()
# Append -ADT to feature names in the ADT assay
adt_counts <- ra_tea_so[["ADT"]]@counts
# rownames(adt_counts) <- paste(rownames(adt_counts), "ADT", sep = "-")
adt_data <- ra_tea_so[["ADT"]]@data
rownames(adt_data) <- rownames(adt_counts)
adt_data %>% head()
adt <- CreateAssayObject(counts = adt_counts)


In [ ]:
# check adt markers names
rownames(adt_counts)


In [ ]:
# creat a trim object to only have the rna and adt data
ra_tea_so_trim <- CreateSeuratObject(ra_tea_so[["RNA"]])
ra_tea_so_trim[["prot"]] <- adt
DefaultAssay(ra_tea_so_trim) <- "prot"
ra_tea_so_trim


In [ ]:
# replace the meate data slot
all(rownames(ra_tea_so_trim@meta.data) == rownames(ra_tea_so@meta.data))
ra_tea_so_trim@meta.data <- ra_tea_so@meta.data


In [ ]:
if (!dir.exists(output_path)) (dir.create(output_path))


In [ ]:
# save the trim seurat data that contains rna + adt to a h5mu data file
WriteH5MU(ra_tea_so_trim, file.path(output_path, "PreRA_teaseq_seurat_qc_filtered_cells_lsi.h5mu"))


## load the ATAC peak matrix


In [ ]:
# load archR data
ra_atac <- loadArchRProject(path = "/home/workspace/data/preRA_teaseq/EXP-00243/atac_arrows")
ra_atac


#### extract the peak matrix and conver it into h5ad


In [ ]:
# extract the peak matrix from archr project
peak_mx <- getMatrixFromProject(ra_atac, "PeakMatrix")


In [ ]:
all.equal(start(ranges(getPeakSet(ra_atac))), start(ranges(rowRanges(peak_mx))))


In [ ]:
# make the peak locatoin as rownames
rownames(peak_mx) <- Signac::GRangesToString(rowRanges(peak_mx), sep = c(":", "-"))


In [ ]:
peak_rowData <- rowData(peak_mx) %>% merge(rowRanges(peak_mx) %>% as_tibble(), sort = FALSE)


In [ ]:
peak_rowData


In [ ]:
# make a sce object to store the preak matrix
peak_sce <- SingleCellExperiment::SingleCellExperiment(list(PeakMatrix = assays(peak_mx)$PeakMatrix),
  colData = colData(peak_mx),
  rowData = peak_rowData
)


In [ ]:
peak_sce


In [ ]:
library(zellkonverter)


In [ ]:
peak_adata


In [ ]:
# convert the peak matrix to a h5ad files
# temp <- tempfile(fileext = file.path(output_path, paste0(proj_name, '_mocha_peakmatrix_l2_celltype.h5ad')))
writeH5AD(peak_sce, file.path(output_path, paste0(proj_name, "_mocha_peakmatrix_l2_celltype.h5ad")))


In [ ]:
sessionInfo()
